# 05 - Model Evaluation and Explainability

This notebook reproduces Module 3's training-only cross-validation and held-out split, evaluates prediction errors and group patterns, and uses permutation importance for global explanations. Correlation and feature importance are predictive associations, not causal effects. A final regression pipeline is saved only if held-out results support it; the risk model is assessed separately.

## 1. Reproducible setup

The same 80/20 seed, preprocessing design, model settings, and cross-validation ranking rules from Module 3 are used so this review corresponds to the saved Module 3 results. Test rows are used for final evaluation and descriptive permutation/group analysis, not to select a different estimator.

In [ ]:
from pathlib import Path
import json, sys, sklearn
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score,
    classification_report, confusion_matrix, f1_score, mean_absolute_error,
    make_scorer, mean_squared_error, precision_score, r2_score, recall_score, roc_auc_score)
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED, TEST_SIZE, FOLDS, RISK_THRESHOLD = 42, 0.20, 5, 2.0
TARGET, ID = "Final_CGPA", "Student_ID"
ROOT=Path.cwd()
if not (ROOT/"data/raw/Student_data.csv").is_file(): ROOT=ROOT.parent
DATA_PATH=ROOT/"data/raw/Student_data.csv"
assert DATA_PATH.is_file(), "Expected data/raw/Student_data.csv relative to the project root."
data=pd.read_csv(DATA_PATH)
features=[c for c in data.columns if c not in {TARGET,ID}]
X=data[features].copy(); y=data[TARGET].astype(float).copy()
num=X.select_dtypes(include=[np.number]).columns.tolist()
cat=X.select_dtypes(exclude=[np.number]).columns.tolist()
assert TARGET not in X and ID not in X and not data.duplicated().any()

def preprocessor():
    nums=Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())])
    cats=Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("encode",OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("numeric",nums,num),("categorical",cats,cat)],remainder="drop")
def pipe(model): return Pipeline([("preprocessor",preprocessor()),("model",model)])

out=ROOT/"reports"/"figures"; out.mkdir(parents=True,exist_ok=True)
print(f"Data: {len(data):,} records; target range {y.min():.2f} to {y.max():.2f}; predictors={len(features)}")
print(f"SHAP installed: {__import__('importlib').util.find_spec('shap') is not None}; permutation importance will be used without adding packages.")

## 2. Reproduce model selection and final predictions

Regression selection is lowest mean five-fold training MAE. Classification selection prioritizes training-fold at-risk recall, then F1 and precision. The risk cutoff remains a provisional assumption (`Final_CGPA < 2.0`), not institutional policy. Each fold fits preprocessing inside its pipeline.

In [ ]:
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=TEST_SIZE,random_state=SEED)
reg_models={"Dummy mean baseline":DummyRegressor(strategy="mean"),
 "Linear Regression":LinearRegression(),"Ridge Regression":Ridge(),
 "Random Forest":RandomForestRegressor(n_estimators=200,random_state=SEED,n_jobs=1),
 "Gradient Boosting":GradientBoostingRegressor(random_state=SEED)}
reg_cv=KFold(n_splits=FOLDS,shuffle=True,random_state=SEED)
rs={"mae":"neg_mean_absolute_error","rmse":"neg_root_mean_squared_error","r2":"r2"}
reg_pipes={}; rows=[]
for name,est in reg_models.items():
 p=pipe(est); reg_pipes[name]=p
 s=cross_validate(p,Xtr,ytr,cv=reg_cv,scoring=rs,n_jobs=1,error_score="raise")
 rows.append({"model":name,"CV MAE":-s["test_mae"].mean(),"CV MAE std":s["test_mae"].std(),
              "CV RMSE":-s["test_rmse"].mean(),"CV R2":s["test_r2"].mean()})
reg_cv_table=pd.DataFrame(rows).sort_values("CV MAE")
reg_name=reg_cv_table.iloc[0].model; reg=reg_pipes[reg_name].fit(Xtr,ytr); pred=reg.predict(Xte)
dummy=reg_pipes["Dummy mean baseline"].fit(Xtr,ytr); dummy_pred=dummy.predict(Xte)
display(reg_cv_table.round(4)); print("Selected regression model:",reg_name)
reg_eval=pd.DataFrame([{"model":"Dummy mean baseline","MAE":mean_absolute_error(yte,dummy_pred),
 "RMSE":np.sqrt(mean_squared_error(yte,dummy_pred)),"R2":r2_score(yte,dummy_pred)},
 {"model":reg_name,"MAE":mean_absolute_error(yte,pred),
 "RMSE":np.sqrt(mean_squared_error(yte,pred)),"R2":r2_score(yte,pred)}])
display(reg_eval.round(4))
assert len(Xtr)==4000 and len(Xte)==1000 and TARGET not in Xtr and ID not in Xtr

## 3. Regression error analysis

Actual-versus-predicted values should cluster around the diagonal. Residuals are `actual minus predicted`; a residual pattern can reveal systematic under/over-prediction or changing error spread. These plots describe this held-out sample and should be read with the numeric metrics.

In [ ]:
residuals=yte.to_numpy()-pred
fig,axes=plt.subplots(1,3,figsize=(17,4.5))
axes[0].scatter(yte,pred,alpha=.45,s=18); axes[0].plot([0,4],[0,4],"k--",lw=1)
axes[0].set(xlabel="Actual Final_CGPA",ylabel="Predicted Final_CGPA",title="Actual vs predicted")
axes[1].scatter(pred,residuals,alpha=.45,s=18); axes[1].axhline(0,color="black",ls="--",lw=1)
axes[1].set(xlabel="Predicted Final_CGPA",ylabel="Residual (actual minus predicted)",title="Residuals vs predictions")
axes[2].hist(residuals,bins=30,edgecolor="white"); axes[2].axvline(0,color="black",ls="--",lw=1)
axes[2].set(xlabel="Residual (CGPA points)",ylabel="Count",title="Residual distribution")
fig.tight_layout(); fig.savefig(out/"regression_diagnostics.png",dpi=160,bbox_inches="tight"); plt.show()
print(f"Residual mean={residuals.mean():.4f}; residual std={residuals.std(ddof=1):.4f}")
print(f"Out of observed range [{y.min():.2f}, {y.max():.2f}]: {int(((pred<y.min())|(pred>y.max())).sum())} of {len(pred)} predictions")

## 4. Performance across groups

We summarize regression MAE by gender and major and classification precision/recall by those groups. Group scores can be unstable when there are few test records or very few at-risk examples. This is a descriptive check, not evidence of fairness or unfairness; interpretation needs larger, representative data and context.

In [ ]:
reg_group_rows=[]
for group_col in ["Gender","Major"]:
 for group,idx in Xte.groupby(group_col).groups.items():
  loc=Xte.index.get_indexer(idx)
  actual=yte.loc[idx].to_numpy(); estimate=pred[loc]
  reg_group_rows.append({"group feature":group_col,"group":group,"n":len(idx),
                         "MAE":mean_absolute_error(actual,estimate),"RMSE":np.sqrt(mean_squared_error(actual,estimate))})
display(pd.DataFrame(reg_group_rows).round(4))

risk=(y<RISK_THRESHOLD).astype(int)
Xct,Xce,yct,yce=train_test_split(X,risk,test_size=TEST_SIZE,random_state=SEED,stratify=risk)
cls_models={"Dummy majority baseline":DummyClassifier(strategy="most_frequent"),
 "Logistic Regression":LogisticRegression(max_iter=2000,class_weight="balanced",random_state=SEED),
 "Random Forest":RandomForestClassifier(n_estimators=200,class_weight="balanced",random_state=SEED,n_jobs=1)}
cv=StratifiedKFold(n_splits=FOLDS,shuffle=True,random_state=SEED)
cs={"precision":make_scorer(precision_score,zero_division=0),"recall":"recall","f1":"f1","roc_auc":"roc_auc"}
cls_pipes={}; crows=[]
for name,est in cls_models.items():
 p=pipe(est); cls_pipes[name]=p
 s=cross_validate(p,Xct,yct,cv=cv,scoring=cs,n_jobs=1,error_score="raise")
 crows.append({"model":name,"CV precision":s["test_precision"].mean(),"CV recall":s["test_recall"].mean(),
               "CV F1":s["test_f1"].mean(),"CV ROC-AUC":s["test_roc_auc"].mean()})
cls_cv_table=pd.DataFrame(crows).sort_values(["CV recall","CV F1","CV precision"],ascending=False,kind="stable")
cls_name=cls_cv_table.iloc[0].model; clf=cls_pipes[cls_name].fit(Xct,yct)
risk_pred=clf.predict(Xce); risk_prob=clf.predict_proba(Xce)[:,1]
print("Risk cutoff is a project assumption: Final_CGPA <",RISK_THRESHOLD)
print("At-risk records:",int(risk.sum()),"of",len(risk),"; held-out positives:",int(yce.sum()),"of",len(yce))
display(cls_cv_table.round(4))
print("Selected classifier:",cls_name)
print(classification_report(yce,risk_pred,labels=[0,1],target_names=["Not at risk","At risk"],zero_division=0))
cm=confusion_matrix(yce,risk_pred,labels=[0,1]); print("Confusion matrix [true rows, predicted columns]:\n",cm)
print("Held-out ROC-AUC:",roc_auc_score(yce,risk_prob) if yce.nunique()==2 else "unavailable; one class")

## 5. Classification plots and group counts

The confusion matrix makes false positives and false negatives visible. ROC shows sensitivity versus false-positive rate across possible probability cutoffs; the reported predictions use the classifier's default 0.5 cutoff. With only a few positive test examples, one case can change recall substantially.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4.5))
ConfusionMatrixDisplay(cm,display_labels=["Not at risk","At risk"]).plot(ax=axes[0],colorbar=False,values_format="d")
axes[0].set_title("Held-out confusion matrix")
if yce.nunique()==2: RocCurveDisplay.from_predictions(yce,risk_prob,ax=axes[1],name=cls_name)
else: axes[1].text(.5,.5,"ROC unavailable: only one true class",ha="center",va="center")
axes[1].plot([0,1],[0,1],"k--",lw=1); axes[1].set_title("Held-out ROC curve")
fig.tight_layout(); fig.savefig(out/"classification_diagnostics.png",dpi=160,bbox_inches="tight"); plt.show()

group_rows=[]
for col in ["Gender","Major"]:
 for label,idx in Xce.groupby(col).groups.items():
  truth=yce.loc[idx].to_numpy(); guess=pd.Series(risk_pred,index=Xce.index).loc[idx].to_numpy()
  group_rows.append({"group feature":col,"group":label,"n":len(idx),"at-risk support":int(truth.sum()),
   "precision":precision_score(truth,guess,zero_division=0),"recall":recall_score(truth,guess,zero_division=0) if truth.sum() else np.nan})
display(pd.DataFrame(group_rows).round(3))

## 6. Global feature importance and association checks

Permutation importance shuffles one original input column at a time and measures the resulting score drop. Because we permute the raw columns through the full pipeline, categories stay grouped under their original field. This is global importance on the held-out sample, not an explanation of an individual prediction. Correlated inputs can share or distort permutation importance; importance is not causation.

SHAP is not installed in the verified environment, so we use scikit-learn permutation importance and do not add a dependency. We separately measure the `Previous_CGPA` / `Final_CGPA` correlation and discuss its strong association as a possible source of target dependence.

In [ ]:
reg_perm=permutation_importance(reg,Xte,yte,n_repeats=15,random_state=SEED,
                                  scoring="neg_mean_absolute_error",n_jobs=1)
reg_imp=pd.DataFrame({"feature":features,"MAE increase mean":reg_perm.importances_mean,
                      "permutation std":reg_perm.importances_std}).sort_values("MAE increase mean",ascending=False)
display(reg_imp.round(4))
fig,ax=plt.subplots(figsize=(8,4.5)); top=reg_imp.sort_values("MAE increase mean")
ax.barh(top.feature,top["MAE increase mean"],xerr=top["permutation std"])
ax.set(xlabel="Increase in held-out MAE when shuffled (higher = more reliance)",title="Regression permutation importance")
fig.tight_layout(); fig.savefig(out/"regression_permutation_importance.png",dpi=160,bbox_inches="tight"); plt.show()

cls_perm=permutation_importance(clf,Xce,yce,n_repeats=15,random_state=SEED,
                                scoring="roc_auc",n_jobs=1)
cls_imp=pd.DataFrame({"feature":features,"ROC-AUC decrease mean":cls_perm.importances_mean,
                      "permutation std":cls_perm.importances_std}).sort_values("ROC-AUC decrease mean",ascending=False)
display(cls_imp.round(4))
fig,ax=plt.subplots(figsize=(8,4.5)); top=cls_imp.sort_values("ROC-AUC decrease mean")
ax.barh(top.feature,top["ROC-AUC decrease mean"],xerr=top["permutation std"])
ax.set(xlabel="Decrease in held-out ROC-AUC when shuffled",title="Risk classifier permutation importance")
fig.tight_layout(); fig.savefig(out/"classification_permutation_importance.png",dpi=160,bbox_inches="tight"); plt.show()
print("Pearson correlation Previous_CGPA vs Final_CGPA:",data[["Previous_CGPA",TARGET]].corr().iloc[0,1])

## 7. Selection decision and artifact handling

Regression is retained only if the selected model has lower held-out MAE and RMSE than the dummy baseline, and its cross-validation and held-out errors are in a similar range. If that check passes, save the complete preprocessing-plus-model pipeline and a small metadata record. `joblib` artifacts can execute code when loaded: only load files you created or otherwise trust.

The academic-risk classifier is **not saved as a final model** in this run: only 42 rows (0.84%) meet the assumed cutoff, leaving eight positive examples in the held-out test set. Its high recall comes with false positives, and performance estimates for such a small minority class are fragile. More representative and policy-grounded data are needed before treating it as validated.

In [ ]:
best=reg_eval.loc[reg_eval.model==reg_name].iloc[0]
base=reg_eval.loc[reg_eval.model=="Dummy mean baseline"].iloc[0]
cv_mae=float(reg_cv_table.loc[reg_cv_table.model==reg_name,"CV MAE"].iloc[0])
regression_supported=bool(best.MAE < base.MAE and best.RMSE < base.RMSE and best.MAE <= 1.5*cv_mae)
print("Regression artifact supported by these within-dataset checks:",regression_supported)
print("CV MAE:",round(cv_mae,4),"; held-out MAE:",round(float(best.MAE),4),"; dummy held-out MAE:",round(float(base.MAE),4))
if regression_supported:
    model_dir=ROOT/"models"; model_dir.mkdir(parents=True,exist_ok=True)
    artifact=model_dir/"final_cgpa_regression.joblib"
    metadata={"task":"Final_CGPA regression","estimator":reg_name,"features":features,
      "excluded_columns":[ID,TARGET],"random_state":SEED,"test_size":TEST_SIZE,
      "cv_folds":FOLDS,"risk_threshold_not_applicable":None,"target_range":[float(y.min()),float(y.max())],
      "cv_mae":cv_mae,"test_mae":float(best.MAE),"test_rmse":float(best.RMSE),"test_r2":float(best["R2"]),
      "python":sys.version.split()[0],"pandas":pd.__version__,"numpy":np.__version__,"scikit_learn":sklearn.__version__}
    joblib.dump({"pipeline":reg,"metadata":metadata},artifact)
    (model_dir/"final_cgpa_regression_metadata.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
    print("Saved trusted local regression pipeline:",artifact.relative_to(ROOT))
else:
    print("No model artifact saved; evaluation did not meet the documented comparison checks.")
print("Figures saved in:",out.relative_to(ROOT))

## 8. Conclusions and limits

The regression pipeline's test results are encouraging only for this dataset and split; synthetic-looking data and strong dependence on `Previous_CGPA` may limit transfer to real cohorts. Review saved figures and group tables rather than relying on one score. The risk classifier is not promoted because the positive class is rare under the provisional cutoff.

Permutation importance describes the model's global reliance on inputs under this evaluation setup. It does not establish cause or explain why one particular student received a prediction. An individual explanation method has not been implemented here. A future evaluation should use independently collected cohorts, an official risk definition, and careful human review.